# FIT5217 Week 7 Lab — Seq2Seq, Attention & Decoding Strategies

This lab is adapted from the [PyTorch Seq2Seq Translation Tutorial](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html).

**What you will learn:**

| Part | Topic | Key Concepts |
|------|-------|-------------|
| 1 | Seq2Seq (no attention) | Encoder–Decoder, GRU, teacher forcing, greedy decoding |
| 2 | Seq2Seq with Attention | Dot-product attention, context vector, soft alignment |
| 3 | Decoding Strategies | Greedy vs. Beam Search vs. Top-k / Top-p sampling |
| 4 | Evaluation & Analysis | Training/validation loss, BLEU score, attention visualisation |
| 5 | Exercises | Fill-in-the-blank coding with hidden solutions |

> **Exercises** are marked with ✏️. Some have hidden solutions — click the *toggle* to reveal.

```
[KEY: > input (French), = target (English), < model output]

> il est en train de peindre un tableau .
= he is painting a picture .
< he is painting a picture .

> vous etes trop maigre .
= you re too skinny .
< you re all alone .        ← the model can make mistakes!
```

### Setup & Dependencies

We import PyTorch and standard libraries. The model runs on GPU if available (significantly faster for training).

In [ ]:
# !pip3 install torch numpy matplotlib

---

In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random
import math
import time

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

**Mount Google Drive** to access the data files:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Upload the data to your google drive and replace the following path according to your Google Drive
file_path = "/content/drive/MyDrive/monashClass/fit5217/weekly_code/week7" 

---
## Part 1 — Seq2Seq Model (no attention)

### 1.1 The Encoder–Decoder Architecture

A **sequence-to-sequence (seq2seq)** model uses two RNNs working together:

<div style="text-align:center; margin:15px 0;">
<img src="https://pytorch.org/tutorials/_images/seq2seq.png" width="500">
<br><em>Figure 1: The encoder reads the source sentence and compresses it into a context vector. The decoder unfolds it into the target sentence.</em>
</div>

**How it works:**

1. **Encoder RNN** — reads the source sentence word-by-word (left to right) and produces a sequence of hidden states $\mathbf{h}_1, \mathbf{h}_2, \ldots, \mathbf{h}_N$. The final hidden state $\mathbf{h}_N$ is the **context vector**.

2. **Decoder RNN** — receives the context vector as its initial hidden state. At each step, it predicts the next target word:
$$P(y_t \mid y_1, \ldots, y_{t-1}, \mathbf{x})$$

This is a **Conditional Language Model**:
- **Language Model** — the decoder generates fluent text in the target language by predicting one word at a time.
- **Conditional** — predictions depend on the source sentence $\mathbf{x}$ (via the context vector from the encoder).

**Key limitation (the bottleneck problem):** The *entire* source sentence must be squeezed into a single fixed-size vector. For long sentences, information gets lost. We solve this in Part 2 with attention.

### 1.2 Background: GRU (Gated Recurrent Unit)

Before we look at the code, let's understand the **GRU** — the RNN variant we use in this lab.

**Problem with vanilla RNNs:** Standard RNNs suffer from the **vanishing gradient problem** — they struggle to learn long-range dependencies because gradients shrink exponentially during backpropagation through many time steps.

**Solution: Gating mechanisms.** The GRU ([Cho et al., 2014](https://aclanthology.org/D14-1179/)) uses two gates to control information flow:

<div style="text-align:center; margin:15px 0;">
<img src="https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-var-GRU.png" width="550">
<br><em>Figure 2: GRU cell architecture. The reset gate (r) controls how much past information to forget; the update gate (z) controls how much new information to let in. (Source: colah.github.io)</em>
</div>

**The two gates:**

| Gate | Symbol | Purpose |
|------|--------|---------|
| **Reset gate** | $r_t = \sigma(W_r [h_{t-1}, x_t])$ | Controls how much of the previous hidden state to *forget* when computing the candidate |
| **Update gate** | $z_t = \sigma(W_z [h_{t-1}, x_t])$ | Controls the *balance* between keeping the old hidden state vs. adopting the new candidate |

**Update equations:**
$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t]) \quad \text{(candidate hidden state)}$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{(final hidden state)}$$

**Why GRU over vanilla RNN?**
- The update gate creates a **direct path** for gradients to flow across many time steps (similar to skip connections), mitigating vanishing gradients.
- It captures **long-range dependencies** far better than a standard RNN.
- Compared to LSTM, GRU has **fewer parameters** (2 gates vs 3) while achieving similar performance on many tasks.

**Why GRU over LSTM?** GRU and LSTM perform similarly on most NLP tasks, but GRU is slightly simpler (fewer parameters, faster training). For this tutorial, GRU is a good balance of simplicity and effectiveness.

### 1.3 Loading & Preprocessing Data

We use English–French translation pairs from the [Tatoeba Project](https://www.manythings.org/anki/).

**Vocabulary representation:** Each language gets a `Lang` object that maps between words and integer indices. Special tokens `SOS` (start-of-sequence, index 0) and `EOS` (end-of-sequence, index 1) mark sentence boundaries.

In [ ]:
SOS_token = 0
EOS_token = 1


class Lang:
    """Vocabulary manager: maps words <-> integer indices."""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # SOS and EOS are pre-defined

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

**Text normalisation:** We convert Unicode to ASCII, lowercase everything, and separate punctuation marks so they become separate tokens.

In [ ]:
def unicodeToAscii(s):
    """Convert Unicode to plain ASCII."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    """Lowercase, trim, remove non-letter characters, separate punctuation."""
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

**Reading and filtering:** We read tab-separated pairs, optionally reversing the direction (we translate French→English, so we reverse the eng-fra file). We filter to short sentences with common prefixes for faster training.

In [ ]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")
    file_name = file_path + '/data/%s-%s.txt' % (lang1, lang2)
    lines = open(file_name, encoding='utf-8').read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)
    return input_lang, output_lang, pairs

**Filter short, simple sentences** for faster training:

In [ ]:
MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

**Prepare data:** Read, filter, and build vocabularies in one call.

In [ ]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(f"  {input_lang.name}: {input_lang.n_words} words")
    print(f"  {output_lang.name}: {output_lang.n_words} words")
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(f"\nExample pair: {random.choice(pairs)}")

**Create a validation set:** We hold out 10% of pairs for monitoring generalisation during training.

In [ ]:
# Split into train / validation
random.shuffle(pairs)
val_size = int(len(pairs) * 0.1)
val_pairs = pairs[:val_size]
train_pairs = pairs[val_size:]
print(f"Training pairs: {len(train_pairs)}, Validation pairs: {len(val_pairs)}")

### 1.4 The Encoder

The encoder processes the source sentence one token at a time:
1. **Embed** the input word index into a dense vector (dimension = `hidden_size`).
2. Pass the embedding + previous hidden state through a **GRU** cell.
3. Output the updated hidden state.

The **final hidden state** after all source tokens becomes the *context vector* — the only information passed to the decoder in the basic model.

<div style="text-align:center; margin:15px 0;">
<img src="https://pytorch.org/tutorials/_images/encoder-network.png" width="300">
<br><em>Figure 3: The encoder RNN processes one word at a time, building up a representation of the source sentence in its hidden state.</em>
</div>

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        # Embedding: word index -> dense vector of size hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        # GRU: processes one step at a time
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        # input: single word index, hidden: previous hidden state
        embedded = self.embedding(input).view(1, 1, -1)  # (1, 1, hidden_size)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden  # output = hidden for GRU (single layer)

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

### 1.5 The Decoder (no attention)

The decoder generates the target sentence one word at a time:
1. Takes an input token (initially `<SOS>`) and the previous hidden state.
2. Passes them through embedding → ReLU → GRU → linear layer → log-softmax.
3. Outputs a **log-probability distribution** over the entire target vocabulary.

**At training time:** We use **teacher forcing** — feeding the *ground-truth* previous word instead of the model's own prediction. This stabilises training but creates **exposure bias** (the model never practises recovering from its own mistakes during training).

**At test time:** The decoder feeds its own predictions back as input (autoregressive generation).

<div style="text-align:center; margin:15px 0;">
<img src="https://pytorch.org/tutorials/_images/decoder-network.png" width="200">
<br><em>Figure 4: The decoder generates one word at a time, using either the ground truth (teacher forcing) or its own prediction as the next input.</em>
</div>

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)  # hidden -> vocab scores
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        # output[0] has shape (1, hidden_size) -> project to vocabulary
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

### 1.6 Training

**Training pipeline:**
1. Run the source sentence through the encoder word-by-word → get hidden states.
2. Initialise the decoder with the encoder's final hidden state (context vector).
3. Feed the decoder step-by-step with teacher forcing (probability 0.5) or its own predictions.
4. Compute **cross-entropy loss** (NLLLoss on log-softmax outputs) at each step.
5. Backpropagate end-to-end through the entire encoder–decoder and update weights.

**Tensor conversion:** We convert sentences to tensors of word indices, appending `EOS` at the end.

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

**The training step function:** Processes one sentence pair. Notice how teacher forcing works — with probability 0.5, we feed the ground-truth target word; otherwise we feed the model's own prediction.

In [ ]:
teacher_forcing_ratio = 0.5

def train_step(input_tensor, target_tensor, encoder, decoder, encoder_optimizer,
               decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    loss = 0

    # --- Encoder: process each source word ---
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    # --- Decoder: generate target words ---
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden  # context vector = encoder's final hidden state

    use_teacher_forcing = random.random() < teacher_forcing_ratio

    if use_teacher_forcing:
        # Feed ground-truth target as next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing
    else:
        # Feed model's own predictions as next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

**Timing helpers:**

In [ ]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / percent
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

**Training loop with validation loss:**

We track both **training loss** (on training pairs) and **validation loss** (on held-out pairs) to monitor overfitting. The validation loss is computed periodically without gradient updates.

In [ ]:
def compute_val_loss(encoder, decoder, val_pairs, criterion, max_length=MAX_LENGTH):
    """Compute average loss on validation set (no gradient updates)."""
    encoder.eval()
    decoder.eval()
    total_loss = 0
    with torch.no_grad():
        for pair in val_pairs:
            input_tensor = tensorFromSentence(input_lang, pair[0])
            target_tensor = tensorFromSentence(output_lang, pair[1])
            encoder_hidden = encoder.initHidden()

            input_length = input_tensor.size(0)
            target_length = target_tensor.size(0)

            for ei in range(input_length):
                _, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

            decoder_input = torch.tensor([[SOS_token]], device=device)
            decoder_hidden = encoder_hidden
            loss = 0

            for di in range(target_length):
                decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
                loss += criterion(decoder_output, target_tensor[di]).item()
                decoder_input = target_tensor[di]  # teacher forcing for consistent eval

            total_loss += loss / target_length

    encoder.train()
    decoder.train()
    return total_loss / len(val_pairs)

**The main training loop** — iterates over random sentence pairs, tracks loss, and periodically computes validation loss:

In [ ]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100,
               learning_rate=0.01, val_every=2000):
    start = time.time()
    train_losses = []
    val_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_samples = [tensorsFromPair(random.choice(train_pairs)) for _ in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        input_tensor, target_tensor = training_samples[iter - 1]
        loss = train_step(input_tensor, target_tensor, encoder, decoder,
                          encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            train_losses.append(plot_loss_avg)
            plot_loss_total = 0

        # Compute validation loss periodically
        if iter % val_every == 0:
            v_loss = compute_val_loss(encoder, decoder, val_pairs, criterion)
            val_losses.append((iter, v_loss))
            print(f"  >> Validation loss: {v_loss:.4f}")

    return train_losses, val_losses

**Plotting training and validation loss:**

In [ ]:
def plot_losses(train_losses, val_losses, plot_every=100):
    """Plot training loss curve and validation loss checkpoints."""
    fig, ax = plt.subplots(figsize=(10, 5))

    # Training loss (one point per plot_every iterations)
    train_x = [(i + 1) * plot_every for i in range(len(train_losses))]
    ax.plot(train_x, train_losses, label='Training Loss', color='steelblue', alpha=0.8)

    # Validation loss (sparse checkpoints)
    if val_losses:
        val_x = [v[0] for v in val_losses]
        val_y = [v[1] for v in val_losses]
        ax.plot(val_x, val_y, 'ro-', label='Validation Loss', markersize=6)

    ax.set_xlabel('Iterations')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 1.7 Evaluation — Greedy Decoding

During evaluation, there are no ground-truth targets. The decoder uses **greedy decoding**: at each step, pick the single highest-probability word (`argmax`) and feed it as the next input.

$$y_t = \arg\max_{w \in V} P(w \mid y_1, \ldots, y_{t-1}, \mathbf{x})$$

**Greedy decoding** is simple and fast, but can make **irreversible errors** — once a wrong word is chosen, all subsequent predictions are conditioned on it with no way to backtrack. We'll explore better alternatives (beam search, sampling) in Part 3.

In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    """Greedy decoding: always pick the top-1 word at each step."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.squeeze().detach()

        return decoded_words

def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words = evaluate(encoder, decoder, pair[0])
        print('<', ' '.join(output_words))
        print('')

### 1.8 Train & Evaluate the Basic Seq2Seq

Let's train the model for 10,000 iterations and observe the training/validation loss curves.

In [ ]:
hidden_size = 128
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Training basic Seq2Seq (no attention)...")
train_losses_basic, val_losses_basic = trainIters(encoder1, decoder1, 10000,
                                                    print_every=500, val_every=500)

**Training and validation loss curves:**

If the validation loss stops decreasing (or increases) while training loss keeps falling, the model is **overfitting** — memorising training examples rather than learning generalisable translation patterns.

In [ ]:
plot_losses(train_losses_basic, val_losses_basic)

**Sample translations (greedy decoding):**

In [ ]:
evaluateRandomly(encoder1, decoder1)

---
## Part 2 — Seq2Seq with Attention

### 2.1 The Bottleneck Problem & How Attention Solves It

In the basic seq2seq model, the decoder only has the encoder's **final hidden state** — one vector that must encode the *entire* source sentence. For long sentences this becomes a bottleneck.

**Attention** lets the decoder *look back* at **all encoder hidden states** at every step:

<div style="text-align:center; margin:15px 0;">
<img src="https://i.imgur.com/1152PYf.png" width="500">
<br><em>Figure 5: The attention mechanism computes a weighted combination of all encoder hidden states, allowing the decoder to focus on different source words at each step.</em>
</div>

**The 4 steps of dot-product attention:**

| Step | Operation | What it computes |
|------|-----------|-----------------|
| 1. **Attention scores** | $e_i^t = \mathbf{s}_t^\top \mathbf{h}_i$ | Similarity between decoder state (query) and each encoder state (key) |
| 2. **Attention distribution** | $\alpha_i^t = \text{softmax}(e_i^t)$ | Probability distribution over source positions |
| 3. **Context vector** | $\mathbf{a}_t = \sum_i \alpha_i^t \, \mathbf{h}_i$ | Weighted summary of the source, focused on relevant parts |
| 4. **Concatenate** | $[\mathbf{a}_t ; \mathbf{s}_t]$ | Combine source context with decoder state for prediction |

**Key intuition:** The decoder state $\mathbf{s}_t$ acts as a **query** ("what do I need to translate next?"), and each encoder state $\mathbf{h}_i$ acts as a **key** ("what information does source position $i$ contain?"). The dot product measures their relevance to each other.

### 2.2 Attention Decoder Implementation

Study the `forward` method carefully — this is where the attention computation happens:
- `torch.bmm(hidden, encoder_outputs.T.unsqueeze(0))` → dot-product attention scores
- `F.softmax(...)` → attention distribution (probabilities summing to 1)
- `torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))` → weighted sum (context vector)
- `torch.cat(...)` → concatenate context vector with decoder hidden state

In [ ]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        # Output takes CONCATENATION of context vector + hidden state (2 * hidden_size)
        self.out = nn.Linear(self.hidden_size * 2, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        # Step 1: Embed input word and update hidden state via GRU
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)
        _, hidden = self.gru(embedded, hidden)

        # Step 2: Compute attention scores (dot product) and distribution (softmax)
        # hidden: (1, 1, H), encoder_outputs.T: (H, max_length)
        attn_weights = F.softmax(
            torch.bmm(hidden, encoder_outputs.T.unsqueeze(0)), dim=-1
        )  # shape: (1, 1, max_length) — attention over each source position

        # Step 3: Compute context vector (weighted sum of encoder outputs)
        attn_output = torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))
        # shape: (1, 1, H) — a step-specific summary of the source

        # Step 4: Concatenate [context_vector ; hidden_state] and predict
        concat_output = torch.cat((attn_output[0], hidden[0]), 1)  # (1, 2*H)
        output = F.log_softmax(self.out(concat_output), dim=1)

        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

### 2.3 Training with Attention

The training loop is identical to Part 1, except the decoder now also receives `encoder_outputs` and returns `attn_weights`.

In [ ]:
def train_step_attn(input_tensor, target_tensor, encoder, decoder, encoder_optimizer,
                    decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden
    use_teacher_forcing = random.random() < teacher_forcing_ratio

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

**Validation loss for attention model:**

In [ ]:
def compute_val_loss_attn(encoder, decoder, val_pairs, criterion, max_length=MAX_LENGTH):
    encoder.eval()
    decoder.eval()
    total_loss = 0
    with torch.no_grad():
        for pair in val_pairs:
            input_tensor = tensorFromSentence(input_lang, pair[0])
            target_tensor = tensorFromSentence(output_lang, pair[1])
            encoder_hidden = encoder.initHidden()
            encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

            for ei in range(input_tensor.size(0)):
                encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
                encoder_outputs[ei] = encoder_output[0, 0]

            decoder_input = torch.tensor([[SOS_token]], device=device)
            decoder_hidden = encoder_hidden
            loss = 0

            for di in range(target_tensor.size(0)):
                decoder_output, decoder_hidden, _ = decoder(
                    decoder_input, decoder_hidden, encoder_outputs)
                loss += criterion(decoder_output, target_tensor[di]).item()
                decoder_input = target_tensor[di]

            total_loss += loss / target_tensor.size(0)

    encoder.train()
    decoder.train()
    return total_loss / len(val_pairs)

**Evaluation and random sampling** (with attention):

In [ ]:
def evaluate_attn(encoder, decoder, sentence, max_length=MAX_LENGTH):
    """Greedy decoding with attention — returns words AND attention weights."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

def evaluateRandomly_attn(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate_attn(encoder, decoder, pair[0])
        print('<', ' '.join(output_words))
        print('')

**Training loop for the attention model** (same structure, different `train_step`):

In [ ]:
def trainIters_attn(encoder, decoder, n_iters, print_every=1000, plot_every=100,
                    learning_rate=0.01, val_every=2000):
    start = time.time()
    train_losses = []
    val_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_samples = [tensorsFromPair(random.choice(train_pairs)) for _ in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        input_tensor, target_tensor = training_samples[iter - 1]
        loss = train_step_attn(input_tensor, target_tensor, encoder, decoder,
                               encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            train_losses.append(plot_loss_total / plot_every)
            plot_loss_total = 0

        if iter % val_every == 0:
            v_loss = compute_val_loss_attn(encoder, decoder, val_pairs, criterion)
            val_losses.append((iter, v_loss))
            print(f"  >> Validation loss: {v_loss:.4f}")

    return train_losses, val_losses

**Train the attention model and plot losses:**

In [ ]:
hidden_size = 128
encoder2 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder = AttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

print("Training Seq2Seq with Attention...")
train_losses_attn, val_losses_attn = trainIters_attn(
    encoder2, attn_decoder, 10000, print_every=500, val_every=500)

**Plot training and validation loss for the attention model:**

In [ ]:
plot_losses(train_losses_attn, val_losses_attn)

**Compare the two loss curves side by side:**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Basic model
ax = axes[0]
train_x = [(i+1)*100 for i in range(len(train_losses_basic))]
ax.plot(train_x, train_losses_basic, label='Train', color='steelblue', alpha=0.8)
if val_losses_basic:
    ax.plot([v[0] for v in val_losses_basic], [v[1] for v in val_losses_basic],
            'ro-', label='Validation', markersize=6)
ax.set_title('Basic Seq2Seq (no attention)')
ax.set_xlabel('Iterations'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True, alpha=0.3)

# Attention model
ax = axes[1]
train_x = [(i+1)*100 for i in range(len(train_losses_attn))]
ax.plot(train_x, train_losses_attn, label='Train', color='steelblue', alpha=0.8)
if val_losses_attn:
    ax.plot([v[0] for v in val_losses_attn], [v[1] for v in val_losses_attn],
            'ro-', label='Validation', markersize=6)
ax.set_title('Seq2Seq with Attention')
ax.set_xlabel('Iterations'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Sample translations from the attention model:**

In [ ]:
evaluateRandomly_attn(encoder2, attn_decoder)

### 2.4 Visualising Attention — Soft Alignment

The attention weights $\boldsymbol{\alpha}^t$ at each decoder step reveal **which source words the decoder is focusing on**. We display this as a heatmap:
- **Rows** = decoder time steps (English output words)
- **Columns** = encoder time steps (French input words)
- **Brighter cells** = higher attention weight

This gives us a **soft alignment** — similar to the hard alignment in Statistical MT, but learned automatically end-to-end.

**What to look for:**
- A roughly **diagonal** pattern suggests word-by-word alignment.
- **Off-diagonal** entries indicate word reordering between languages.
- **Spread** attention (many columns lit up) suggests the model is uncertain about which source word to attend to.

In [ ]:
def showAttention(input_sentence, output_words, attentions):
    fig, ax = plt.subplots(figsize=(8, 6))
    cax = ax.matshow(attentions.numpy(), cmap='viridis')
    fig.colorbar(cax, ax=ax, shrink=0.8)

    ax.set_xticklabels([''] + input_sentence.split(' ') + ['<EOS>'], rotation=45, ha='left')
    ax.set_yticklabels([''] + output_words)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.set_xlabel('Source (French)')
    ax.set_ylabel('Target (English)')
    ax.set_title('Attention Weights (Soft Alignment)')
    plt.tight_layout()
    plt.show()

def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate_attn(encoder2, attn_decoder, input_sentence)
    print(f'input  = {input_sentence}')
    print(f'output = {" ".join(output_words)}')
    showAttention(input_sentence, output_words, attentions)

**Let's visualise attention for several sentences:**

In [ ]:
evaluateAndShowAttention("elle a cinq ans .")
evaluateAndShowAttention("je suis trop froid .")
evaluateAndShowAttention("il est un jeune directeur .")

### 2.5 Discussion: How Do We Evaluate Machine Translation?

We've been eyeballing translations — but how do we measure quality **automatically**?

**BLEU Score** (Bilingual Evaluation Understudy) is the most common MT metric:
- Compares machine output to one or more human reference translations.
- Computes **n-gram precision** (how many n-grams in the output appear in the reference).
- Adds a **brevity penalty** for translations that are too short.

$$\text{BLEU} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} \frac{1}{N} \log p_n\right)$$

where $p_n$ is the n-gram precision and BP penalises short translations.

**BLEU limitations:**
- Only counts **exact n-gram matches** — valid synonyms ("car" vs "automobile") score 0.
- A good translation with different word choices can get a low BLEU score.
- More recent metrics (METEOR, BERTScore, COMET) use semantic similarity instead of exact matches.

Let's compute a simple BLEU score for our model:

In [ ]:
from collections import Counter

def simple_bleu(reference, candidate, max_n=4):
    """Compute a simple sentence-level BLEU score (without smoothing)."""
    ref_words = reference.split()
    cand_words = [w for w in candidate if w != '<EOS>']

    if len(cand_words) == 0:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter([tuple(ref_words[i:i+n]) for i in range(len(ref_words)-n+1)])
        cand_ngrams = Counter([tuple(cand_words[i:i+n]) for i in range(len(cand_words)-n+1)])

        if len(cand_ngrams) == 0:
            precisions.append(0)
            continue

        matches = sum((cand_ngrams & ref_ngrams).values())
        precisions.append(matches / max(sum(cand_ngrams.values()), 1))

    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref_words) / max(len(cand_words), 1)))

    # Geometric mean of precisions (with smoothing to avoid log(0))
    log_avg = sum(math.log(max(p, 1e-10)) for p in precisions) / max_n
    return bp * math.exp(log_avg)

# Evaluate BLEU on some pairs
print("BLEU scores (attention model):")
print("-" * 60)
bleu_scores = []
for pair in random.sample(pairs, 20):
    output_words, _ = evaluate_attn(encoder2, attn_decoder, pair[0])
    score = simple_bleu(pair[1], output_words)
    bleu_scores.append(score)
    print(f"  Ref: {pair[1]:<35} Out: {' '.join(output_words):<30} BLEU: {score:.3f}")

print(f"\nAverage BLEU: {np.mean(bleu_scores):.3f}")

---
## Part 3 — Decoding Strategies

So far we've only used **greedy decoding**. Let's implement and compare four strategies:

| Strategy | Mechanism | Best for |
|----------|-----------|----------|
| **Greedy** | Pick argmax at each step | Fast baseline |
| **Beam Search** | Keep top-$k$ hypotheses | Translation (high accuracy) |
| **Top-$k$ Sampling** | Sample from top-$k$ words | Creative generation |
| **Top-$p$ (Nucleus)** | Sample from smallest set with cumulative prob $\geq p$ | Adaptive creative generation |

<div style="text-align:center; margin:15px 0;">
<img src="https://huggingface.co/blog/assets/02_how-to-generate/beam_search.png" width="500">
<br><em>Figure 6: Beam search with k=2 keeps two hypotheses at each step, allowing it to find better sequences than greedy decoding. (Source: HuggingFace)</em>
</div>

### 3.1 Beam Search

**Beam search** keeps the top-$k$ (beam width) partial hypotheses at each step:

1. **Expand** each of the $k$ hypotheses by every word in the vocabulary.
2. **Score** each candidate by cumulative log-probability.
3. **Keep** only the top-$k$ candidates.
4. When a hypothesis generates `<EOS>`, set it aside as completed.
5. Finally, select the hypothesis with the best **length-normalised score**:

$$\text{score}_{\text{norm}} = \frac{1}{T} \sum_{t=1}^{T} \log P(y_t \mid y_1, \ldots, y_{t-1}, \mathbf{x})$$

Length normalisation prevents bias toward shorter sequences (which accumulate fewer negative log-prob terms).

In [ ]:
def beam_search_attn(encoder, decoder, sentence, beam_width=3, max_length=MAX_LENGTH):
    """Beam search decoding with the attention model."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        # Each hypothesis: (cumulative_log_prob, [word_indices], decoder_hidden, [attentions])
        beams = [(0.0, [SOS_token], encoder_hidden, [])]
        completed = []

        for step in range(max_length):
            all_candidates = []
            for score, seq, hidden, attn_list in beams:
                if seq[-1] == EOS_token:
                    completed.append((score, seq, hidden, attn_list))
                    continue

                dec_input = torch.tensor([[seq[-1]]], device=device)
                dec_output, dec_hidden, dec_attn = decoder(dec_input, hidden, encoder_outputs)
                log_probs = dec_output.squeeze(0)
                topk_log_probs, topk_indices = log_probs.topk(beam_width)

                for i in range(beam_width):
                    new_score = score + topk_log_probs[i].item()
                    new_seq = seq + [topk_indices[i].item()]
                    new_attn = attn_list + [dec_attn.squeeze().cpu()]
                    all_candidates.append((new_score, new_seq, dec_hidden, new_attn))

            if not all_candidates:
                break
            all_candidates.sort(key=lambda x: x[0], reverse=True)
            beams = all_candidates[:beam_width]

        completed.extend(beams)
        best = max(completed, key=lambda x: x[0] / max(len(x[1]) - 1, 1))
        _, best_seq, _, best_attn = best

        decoded_words = []
        for idx in best_seq[1:]:  # skip SOS
            if idx == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx])

        attn_matrix = torch.stack(best_attn) if best_attn else torch.zeros(1, max_length)
        return decoded_words, attn_matrix

**Compare greedy vs beam search:**

In [ ]:
test_sentences = [
    "elle a cinq ans .",
    "je suis trop froid .",
    "il est un jeune directeur .",
    "nous sommes en retard .",
    "vous etes tres gentil .",
]

print(f"{'Source':<35} {'Greedy':<25} {'Beam (k=3)':<25}")
print("=" * 85)
for sent in test_sentences:
    greedy_words, _ = evaluate_attn(encoder2, attn_decoder, sent)
    beam_words, _ = beam_search_attn(encoder2, attn_decoder, sent, beam_width=3)
    print(f"{sent:<35} {' '.join(greedy_words):<25} {' '.join(beam_words):<25}")

### 3.2 Top-$k$ Sampling

Instead of always picking the best word, **top-$k$ sampling** randomly samples from the $k$ most probable words (after re-normalising). This introduces diversity — useful for creative generation.

**Limitation:** $k$ is fixed regardless of distribution shape. If the model is very confident (one word at 99%), $k$ words include many unlikely choices.

#### ✏️ Exercise 1: Implement Top-$k$ Sampling

Complete the `top_k_sample` function:
1. Get the top-$k$ log-probabilities and their indices.
2. Convert to probabilities (`torch.exp`).
3. Re-normalise so they sum to 1.
4. Sample using `torch.multinomial`.

In [ ]:
def top_k_sample(log_probs, k):
    """Sample from the top-k most probable words."""
    # ===================== YOUR CODE HERE =====================
    # Step 1: Get top-k log-probs and indices
    topk_log_probs, topk_indices = None, None  # TODO

    # Step 2: Convert to probabilities
    topk_probs = None  # TODO

    # Step 3: Re-normalise
    topk_probs = None  # TODO

    # Step 4: Sample one index
    sampled_idx = None  # TODO
    # ==========================================================
    word_index = topk_indices[sampled_idx]
    return word_index.item(), topk_log_probs[sampled_idx].item()


def topk_decode_attn(encoder, decoder, sentence, k=5, max_length=MAX_LENGTH):
    """Decode using top-k sampling."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_tensor.size(0)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            word_idx, _ = top_k_sample(decoder_output.squeeze(0), k)
            if word_idx == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[word_idx])
            decoder_input = torch.tensor([[word_idx]], device=device)

        return decoded_words

<details>
<summary>🔑 <b>Click to reveal solution</b></summary>

```python
def top_k_sample(log_probs, k):
    topk_log_probs, topk_indices = log_probs.topk(k)
    topk_probs = torch.exp(topk_log_probs)
    topk_probs = topk_probs / topk_probs.sum()
    sampled_idx = torch.multinomial(topk_probs, 1).item()
    word_index = topk_indices[sampled_idx]
    return word_index.item(), topk_log_probs[sampled_idx].item()
```
</details>

### 3.3 Top-$p$ (Nucleus) Sampling

**Top-$p$ sampling** adapts the candidate set to the distribution shape:
1. Sort words by probability (descending).
2. Include words until cumulative probability reaches $p$.
3. Re-normalise and sample.

**Key advantage over top-$k$:** The nucleus size adapts automatically — small when confident, large when uncertain.

#### ✏️ Exercise 2: Implement Top-$p$ Sampling

Complete the `top_p_sample` function.

In [ ]:
def top_p_sample(log_probs, p=0.9):
    """Sample from the nucleus (smallest set with cumulative prob >= p)."""
    probs = torch.exp(log_probs)

    # ===================== YOUR CODE HERE =====================
    # Step 1: Sort probabilities descending
    sorted_probs, sorted_indices = None, None  # TODO

    # Step 2: Cumulative sum
    cumulative_probs = None  # TODO

    # Step 3: Create mask for words to remove (beyond nucleus)
    sorted_indices_to_remove = None  # TODO: boolean mask where cumsum > p
    sorted_indices_to_remove[0] = False  # always keep at least one word

    # Step 4: Zero out removed words and re-normalise
    sorted_probs[sorted_indices_to_remove] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum()

    # Step 5: Sample
    sampled_pos = None  # TODO
    # ==========================================================

    sampled_original_idx = sorted_indices[sampled_pos]
    return sampled_original_idx.item(), log_probs[sampled_original_idx].item()


def topp_decode_attn(encoder, decoder, sentence, p=0.9, max_length=MAX_LENGTH):
    """Decode using top-p (nucleus) sampling."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_tensor.size(0)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            word_idx, _ = top_p_sample(decoder_output.squeeze(0), p)
            if word_idx == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[word_idx])
            decoder_input = torch.tensor([[word_idx]], device=device)

        return decoded_words

<details>
<summary>🔑 <b>Click to reveal solution</b></summary>

```python
def top_p_sample(log_probs, p=0.9):
    probs = torch.exp(log_probs)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[0] = False
    sorted_probs[sorted_indices_to_remove] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum()
    sampled_pos = torch.multinomial(sorted_probs, 1).item()
    sampled_original_idx = sorted_indices[sampled_pos]
    return sampled_original_idx.item(), log_probs[sampled_original_idx].item()
```
</details>

### 3.4 Compare All Decoding Strategies

Let's run all four strategies on the same inputs. Sampling methods are run multiple times to show variability.

In [ ]:
def compare_decoders(sentence, n_samples=3):
    print(f"\nSource: {sentence}")
    print("-" * 70)

    greedy_words, _ = evaluate_attn(encoder2, attn_decoder, sentence)
    print(f"  Greedy:          {' '.join(greedy_words)}")

    beam_words, _ = beam_search_attn(encoder2, attn_decoder, sentence, beam_width=3)
    print(f"  Beam (k=3):      {' '.join(beam_words)}")

    for i in range(n_samples):
        topk_words = topk_decode_attn(encoder2, attn_decoder, sentence, k=5)
        print(f"  Top-k (k=5) #{i+1}:  {' '.join(topk_words)}")

    for i in range(n_samples):
        topp_words = topp_decode_attn(encoder2, attn_decoder, sentence, p=0.9)
        print(f"  Top-p (p=.9) #{i+1}: {' '.join(topp_words)}")

for sent in test_sentences:
    compare_decoders(sent)

---
## Part 4 — Further Exercises

#### ✏️ Exercise 3: Effect of Beam Width

Run beam search with widths $k \in \{1, 2, 3, 5, 10\}$. Answer:
1. How does $k=1$ compare to greedy?
2. At what $k$ do you see diminishing returns?
3. Can large $k$ ever produce *worse* translations?

In [ ]:
# Your code here
for k in [1, 2, 3, 5, 10]:
    sent = "elle a cinq ans ."
    words, _ = beam_search_attn(encoder2, attn_decoder, sent, beam_width=k)
    print(f"  Beam k={k:2d}: {' '.join(words)}")

#### ✏️ Exercise 4: Effect of Top-$p$ Threshold

Generate 5 samples each for $p \in \{0.5, 0.7, 0.9, 0.95, 1.0\}$. Observe how diversity changes.

In [ ]:
sent = "je suis trop froid ."
for p_val in [0.5, 0.7, 0.9, 0.95, 1.0]:
    print(f"\n  p = {p_val}:")
    for i in range(5):
        words = topp_decode_attn(encoder2, attn_decoder, sent, p=p_val)
        print(f"    #{i+1}: {' '.join(words)}")

#### ✏️ Exercise 5: Attention Analysis

1. Generate attention heatmaps for 3 sentences of your choice.
2. Do the patterns look roughly diagonal?
3. Can you find cases where word reordering is visible in the attention?

In [ ]:
# Your code here
evaluateAndShowAttention("il est un jeune directeur .")

#### ✏️ Exercise 6: Compare With vs Without Attention

Translate the same sentences with both models. Which produces better results?

In [ ]:
test_sents = ["elle a cinq ans .", "je suis trop froid .", "nous sommes en retard ."]
print(f"{'Source':<30} {'No Attention':<25} {'With Attention':<25}")
print("=" * 80)
for sent in test_sents:
    no_attn = ' '.join(evaluate(encoder1, decoder1, sent))
    with_attn = ' '.join(evaluate_attn(encoder2, attn_decoder, sent)[0])
    print(f"{sent:<30} {no_attn:<25} {with_attn:<25}")

#### ✏️ Exercise 7 (Extension): Temperature Scaling

Divide logits by temperature $\tau$ before sampling: $P(w) = \text{softmax}(z / \tau)$
- $\tau < 1$: sharper → more confident
- $\tau > 1$: flatter → more diverse

In [ ]:
# Your code here
def temperature_sample(log_probs, temperature=1.0):
    adjusted = log_probs / temperature
    probs = torch.exp(adjusted)
    probs = probs / probs.sum()
    sampled_idx = torch.multinomial(probs, 1).item()
    return sampled_idx, log_probs[sampled_idx].item()

# TODO: Write a temperature_decode_attn function and test with tau in [0.5, 1.0, 1.5, 2.0]

<details>
<summary>🔑 <b>Click to reveal full solution</b></summary>

```python
def temperature_decode_attn(encoder, decoder, sentence, temperature=1.0, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_tensor.size(0)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            word_idx, _ = temperature_sample(decoder_output.squeeze(0), temperature)
            if word_idx == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[word_idx])
            decoder_input = torch.tensor([[word_idx]], device=device)
        return decoded_words

sent = "je suis trop froid ."
for tau in [0.5, 1.0, 1.5, 2.0]:
    print(f"\n  Temperature = {tau}:")
    for i in range(3):
        words = temperature_decode_attn(encoder2, attn_decoder, sent, temperature=tau)
        print(f"    #{i+1}: {' '.join(words)}")
```
</details>

---
### Glossary

| Concept | What it does |
|---------|-------------|
| **GRU** | Gated RNN variant that mitigates vanishing gradients with reset/update gates |
| **Encoder** | Reads source sentence → sequence of hidden states |
| **Decoder** | Generates target words conditioned on encoder output |
| **Teacher forcing** | Feed ground-truth (not predictions) during training; causes exposure bias |
| **Bottleneck** | Single context vector cannot capture long source sentences |
| **Attention** | Decoder looks at all encoder states via weighted sum (query–key–value) |
| **Greedy decoding** | Pick argmax at each step (fast but commits to errors) |
| **Beam search** | Keep top-$k$ hypotheses (better quality, more compute) |
| **Top-$k$ sampling** | Sample from top-$k$ words (fixed diversity) |
| **Top-$p$ sampling** | Sample from nucleus (adaptive diversity) |
| **Temperature** | Scale logits to control sharpness of distribution |
| **BLEU score** | Automatic MT evaluation via n-gram precision against references |

# References
- [Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation](https://aclanthology.org/D14-1179/) (Cho et al., EMNLP 2014)